# Análisis y propuesta de mejora VECTRA

**Proyecto:** VECTRA — Sistema IoT para monitoreo vehicular

***ELLIOT GONZALEZ BALLESTEROS***

## Objetivo

Desarrollar una segunda propuesta de mejora para la organización del software de VECTRA, tomando como base el análisis realizado previamente por el equipo.

Esta versión propone organizar el sistema mediante una arquitectura por capas y definir interfaces entre los componentes principales, con el objetivo de reducir dependencias y facilitar futuras modificaciones.

## Diferencia respecto a la propuesta A

La propuesta A se enfoca principalmente en dividir las responsabilidades en clases especializadas.

La propuesta B mantiene esa separación, pero además organiza las clases en capas y propone interfaces para evitar que los componentes dependan directamente de implementaciones concretas.

## 1. Enfoque de la propuesta B

La propuesta B organiza VECTRA en capas de software. La intención es que cada grupo de componentes tenga una función específica y que las dependencias sigan una dirección clara.

La estructura propuesta se divide en cuatro partes:

- **Presentación:** muestra la información al usuario.
- **Aplicación y negocio:** coordina recorridos, análisis, alertas y reglas del sistema.
- **Persistencia:** se encarga de guardar y consultar información.
- **Infraestructura:** maneja sensores, ESP32 y comunicación con servicios externos.

Además, se propone utilizar interfaces en algunos componentes importantes. Esto permitiría cambiar una implementación sin modificar directamente las clases que dependen de ella.

Por ejemplo, el análisis de anomalías podría utilizar una interfaz llamada `AnomalyDetector`. En el futuro sería posible cambiar el algoritmo de detección sin modificar la lógica encargada de gestionar los recorridos.


## 2. Clases e interfaces propuestas

Para organizar VECTRA por capas, se propone dividir las responsabilidades de la siguiente manera.

### 2.1 Capa de presentación

**`DashboardController`**

Responsabilidad:
- Recibir solicitudes de la interfaz.
- Solicitar información de recorridos, alertas y análisis.
- Entregar al dashboard los datos que necesita mostrar.

Esta clase no debería acceder directamente a sensores ni a la base de datos.

### 2.2 Capa de aplicación y negocio

**`TripService`**

Responsabilidad:
- Iniciar y finalizar recorridos.
- Asociar mediciones con un recorrido.
- Consultar la información necesaria para trabajar con viajes.

**`TelemetryService`**

Responsabilidad:
- Coordinar el procesamiento de las mediciones recibidas.
- Validar la información antes de almacenarla.
- Enviar la telemetría a los componentes que la necesitan.

**`AlertService`**

Responsabilidad:
- Recibir el resultado del análisis.
- Decidir si debe generarse una alerta.
- Preparar la información que posteriormente se mostrará al usuario.

### 2.3 Interfaces de negocio

**`AnomalyDetector`**

Responsabilidad:
- Definir una operación común para analizar telemetría y detectar posibles desviaciones.

La ventaja de utilizar una interfaz es que el sistema no tendría que depender de un algoritmo específico.

Por ejemplo, podrían existir diferentes implementaciones:

- `StatisticalAnomalyDetector`
- `MachineLearningAnomalyDetector`

Ambas cumplirían el mismo contrato definido por `AnomalyDetector`.

### 2.4 Capa de persistencia

**`TelemetryRepository`**

Responsabilidad:
- Guardar mediciones.
- Consultar mediciones por recorrido.
- Recuperar históricos.

**`TripRepository`**

Responsabilidad:
- Guardar información de los recorridos.
- Consultar recorridos registrados.

Estas responsabilidades se mantienen separadas de la lógica de negocio para evitar que los servicios dependan directamente de una base de datos específica.

### 2.5 Capa de infraestructura

**`SensorReader`**

Responsabilidad:
- Obtener las mediciones de los sensores físicos.

**`TelemetrySender`**

Responsabilidad:
- Definir la forma en que las mediciones son enviadas hacia la plataforma.

Una implementación podría ser:

`WifiTelemetrySender`

En el futuro podría reemplazarse por otra forma de comunicación sin cambiar directamente la lógica del sistema.

## 2.6 Resumen de responsabilidades

| Componente | Responsabilidad principal |
|---|---|
| `DashboardController` | Comunicación entre la interfaz y los servicios |
| `TripService` | Gestión de recorridos |
| `TelemetryService` | Coordinación del procesamiento de telemetría |
| `AlertService` | Gestión de alertas |
| `AnomalyDetector` | Contrato para detectar anomalías |
| `TelemetryRepository` | Persistencia de mediciones |
| `TripRepository` | Persistencia de recorridos |
| `SensorReader` | Lectura de sensores |
| `TelemetrySender` | Contrato para enviar telemetría |

Con esta organización, cada componente tiene una responsabilidad más definida y las dependencias entre las partes del sistema pueden controlarse con mayor claridad.

## 3. Diagrama del diseño mejorado

El siguiente diagrama representa la organización propuesta mediante capas e interfaces.

```mermaid
flowchart TD

    subgraph PRESENTACION["Capa de presentación"]
        A[Dashboard]
        B[DashboardController]
        A --> B
    end

    subgraph NEGOCIO["Capa de aplicación y negocio"]
        C[TripService]
        D[TelemetryService]
        E[AlertService]
        F["AnomalyDetector <<interface>>"]
    end

    subgraph PERSISTENCIA["Capa de persistencia"]
        G[TelemetryRepository]
        H[TripRepository]
    end

    subgraph INFRAESTRUCTURA["Capa de infraestructura"]
        I[SensorReader]
        J["TelemetrySender <<interface>>"]
        K[WifiTelemetrySender]
    end

    B --> C
    B --> D
    B --> E

    D --> F
    D --> G
    C --> H
    E --> F

    I --> D
    D --> J
    K -. implementa .-> J
```

### Interpretación del diagrama

La interfaz se comunica con `DashboardController`, que sirve como punto de entrada hacia los servicios.

La lógica principal se mantiene dentro de `TripService`, `TelemetryService` y `AlertService`.

Los servicios utilizan repositorios para almacenar y consultar información, en lugar de acceder directamente a la base de datos.

Las interfaces `AnomalyDetector` y `TelemetrySender` permiten que el sistema dependa de contratos y no de una implementación concreta.

Esto haría posible cambiar el algoritmo de análisis o el mecanismo de comunicación con un impacto menor en otras partes del sistema.

## 4. Comparación entre la propuesta A y la propuesta B

La propuesta A y la propuesta B buscan mejorar la organización del software de VECTRA, pero utilizan enfoques diferentes.

La propuesta A se enfoca principalmente en separar las responsabilidades en clases especializadas, de manera que cada clase tenga una función concreta.

La propuesta B toma esa misma necesidad de separación, pero agrega una organización por capas y el uso de interfaces para reducir la dependencia entre implementaciones.

| Aspecto | Propuesta A | Propuesta B |
|---|---|---|
| Organización principal | Clases especializadas | Capas de software |
| Enfoque | Separación de responsabilidades | Separación de responsabilidades y control de dependencias |
| Comunicación entre componentes | Relaciones directas entre clases | Uso de servicios e interfaces |
| Persistencia | `TelemetryRepository` centraliza acceso a datos | Se separan `TelemetryRepository` y `TripRepository` |
| Detección de anomalías | `AnomalyDetectionService` | Interfaz `AnomalyDetector` con diferentes implementaciones posibles |
| Comunicación | `CommunicationGateway` | Interfaz `TelemetrySender` |
| Presentación | `DashboardService` | `DashboardController` separado de la lógica |
| Extensibilidad | Permite agregar nuevas clases | Permite cambiar implementaciones sin modificar otros componentes |

### 4.1 Diferencia principal

La diferencia más importante es que en la propuesta B algunas partes del sistema no dependen directamente de una clase concreta.

Por ejemplo, `TelemetryService` puede trabajar con la interfaz `AnomalyDetector` sin importar si el análisis se realiza con un método estadístico o con un modelo de aprendizaje automático.

De la misma manera, el sistema puede utilizar `TelemetrySender` sin depender directamente de Wi-Fi. Si en el futuro se utiliza otra tecnología de comunicación, podría añadirse una nueva implementación sin modificar la lógica principal.

Por esta razón, la propuesta B no reemplaza completamente la idea de la propuesta A, sino que la reorganiza y agrega una separación más clara entre las capas del sistema.

## 5. Beneficios de la propuesta B

La organización por capas y el uso de interfaces pueden aportar varias ventajas al desarrollo de VECTRA.

### Mantenimiento

Al separar presentación, lógica, persistencia e infraestructura, los cambios pueden realizarse en una parte específica del sistema sin modificar directamente las demás.

Por ejemplo, un cambio en la interfaz del dashboard no debería afectar la forma en que se almacenan las mediciones.

### Reutilización

Las interfaces permiten reutilizar los mismos servicios con diferentes implementaciones.

Por ejemplo, `AnomalyDetector` puede ser utilizado por distintos algoritmos de detección sin cambiar el servicio que solicita el análisis.

### Pruebas

Los componentes pueden probarse de manera independiente.

Para probar `TelemetryService` no sería necesario contar con sensores físicos ni conexión Wi-Fi, ya que podrían utilizarse implementaciones de prueba de `SensorReader`, `TelemetrySender` o `AnomalyDetector`.

### Incorporación de nuevas funcionalidades

La estructura facilita agregar nuevas opciones sin modificar gran parte del sistema.

Algunos ejemplos podrían ser:

- Nuevos algoritmos de detección de anomalías.
- Nuevos mecanismos de comunicación.
- Nuevas fuentes de datos.
- Nuevos tipos de reportes.
- Nuevas interfaces de usuario.

### Menor acoplamiento

El uso de interfaces reduce la dependencia directa entre componentes.

Esto permite que las distintas partes del sistema puedan cambiar de forma más independiente y facilita mantener una estructura más ordenada a medida que VECTRA crece.

## 6. Referencia del diseño original

Para comparar la propuesta B con la estructura actual de VECTRA, se toma como referencia la arquitectura conceptual definida previamente por el equipo.

El flujo general del sistema puede resumirse de la siguiente manera:

```mermaid
flowchart LR
    A[Vehículo] --> B[Sensores]
    B --> C[ESP32]
    C --> D[Comunicación Wi-Fi]
    D --> E[Almacenamiento]
    E --> F[Procesamiento]
    F --> G[Análisis de anomalías]
    G --> H[Alertas]
    E --> I[Históricos]
    E --> J[Gestión de recorridos]
    H --> K[Dashboard]
    I --> K
    J --> K
```

En esta estructura se identifican responsabilidades relacionadas con:

- Lectura de sensores.
- Comunicación de datos.
- Almacenamiento de telemetría.
- Procesamiento de información.
- Detección de anomalías.
- Gestión de alertas.
- Gestión de recorridos.
- Visualización mediante dashboard.

Como VECTRA se encuentra definido principalmente a nivel conceptual, este diagrama representa los componentes y responsabilidades descritos por el proyecto y no una implementación completa de clases existente.

## 7. Conclusión

La propuesta B busca organizar el software de VECTRA mediante una estructura por capas y el uso de interfaces en los puntos donde podrían existir diferentes implementaciones.

A diferencia de la propuesta A, que se enfoca principalmente en dividir las responsabilidades en clases especializadas, esta versión también busca controlar la manera en que esas clases dependen unas de otras.

El uso de componentes como `AnomalyDetector` y `TelemetrySender` permitiría cambiar el algoritmo de análisis o el mecanismo de comunicación sin modificar directamente la lógica principal del sistema.

También se propone separar claramente la presentación, la lógica de negocio, la persistencia y la infraestructura. Esto facilitaría identificar dónde debe realizarse cada cambio y permitiría probar los componentes de manera más independiente.

Esta propuesta no busca reemplazar completamente la estructura planteada en la versión A, sino presentar una alternativa de organización que prioriza la separación por capas y la reducción del acoplamiento.

Al igual que la propuesta anterior, se trata de un diseño conceptual. Su efectividad tendría que comprobarse posteriormente durante la implementación del sistema.